# Memory with Tools (Memory-as-a-Tool)

> **Give the agent callable memory tools (`save_memory`, `search_memory`, `update_memory`, `delete_memory`) and let it decide when and what to remember.**

Think of a librarian who decides what books to shelve, which ones to look up for you, and which outdated ones to discard. The librarian doesn't follow a fixed routine. They use their judgment based on what you ask. Memory-as-a-Tool gives an AI agent the same kind of control over its own memory. The analogy breaks down in one way: the agent also writes the books (it creates facts from conversations).

Most agent memory systems operate invisibly. A framework intercepts every message, embeds it, and stuffs retrieved chunks into the prompt. The agent has no say in what gets stored. It can't update stale facts. It can't choose to forget.

This works for basic chatbots. But it breaks down when the agent needs to be selective. For example: storing a user's dietary restriction while ignoring small talk. Or correcting a previously saved fact.

The **Memory-as-a-Tool** pattern flips the model. Memory operations become tool schemas (structured descriptions of callable functions). The LLM decides during generation whether to call `save_memory`, `search_memory`, `update_memory`, or `delete_memory`. This uses the same tool-use loop that powers every modern agent. The memory tools sit alongside other tools like web search or code execution.

This notebook shows you how to build it from scratch with the **Anthropic SDK** (Anthropic's Python library for calling Claude). You'll define four memory tools, build a vector-backed memory store, and wire up the agentic tool-use loop.

**By the end you'll understand:**
- How to define memory operations as tool schemas.
- How to build a memory store with embedding-based semantic search.
- How the agentic loop dispatches tool calls and feeds results back.
- When agent-driven memory beats hidden pipeline memory.


## Key Concepts

- **Tool schema**: A structured description of a callable function. It includes the tool's name, what it does, and what inputs it expects (as a JSON Schema). The LLM reads these schemas and decides which tools to call.
- **Memory CRUD**: The four core operations for managing stored data. CRUD stands for Create, Read, Update, Delete. Here we expose them as `save_memory`, `search_memory`, `update_memory`, and `delete_memory`.
- **Agentic tool-use loop**: The cycle where the LLM generates a response, optionally requests a tool call, receives the result, and continues generating. This loop repeats until the model produces a final text response with no tool calls.
- **`tool_use` / `tool_result` blocks**: In Anthropic's API, when the model wants to call a tool, it emits a `tool_use` content block. Your code executes the tool and returns a `tool_result` block. These blocks sit inside the normal message list.
- **Embedding**: A list of numbers (a vector) that captures the meaning of a piece of text. Similar texts produce vectors that point in similar directions. We use embeddings to find memories that match a search query.
- **Cosine similarity**: A way to measure how similar two vectors are. It computes the cosine of the angle between them. A value of 1.0 means identical direction. A value of 0.0 means unrelated.
- **Semantic search**: Finding stored items by meaning rather than exact keyword match. We embed the query, then find stored memories whose embeddings have the highest cosine similarity.
- **Agent-driven memory**: The LLM decides on its own when to store, retrieve, modify, or discard information. No external heuristics force memory writes on every turn.


## Architecture

<p align="center">
  <img src="../../images/diagrams/23_memory_with_tools.svg" alt="Memory with Tools architecture diagram" width="720"/>
</p>

The diagram shows the full flow:

1. The user sends a message. The system prompt includes tool definitions for all four memory tools.
2. The LLM generates a response. If it decides memory is relevant, it emits a `tool_use` block (for example, `search_memory({"query": "dietary restrictions"})`).
3. Your application executes the tool call against the memory store and returns a `tool_result` block.
4. The LLM sees the result. It can make another tool call (chaining) or produce a final text response.
5. After responding, the LLM may also call `save_memory` to store new facts. All of this happens within the same turn.


## Setup

Install dependencies and configure API access. You'll need an `ANTHROPIC_API_KEY` environment variable (your secret key for calling the Anthropic API). We also use `numpy` for cosine similarity calculations in the memory store.


In [ ]:
%pip install -q anthropic python-dotenv numpy


Import the Anthropic SDK and standard library helpers. The API key loads from a `.env` file.


In [ ]:
import os
import json
import uuid
from datetime import datetime, timezone

import numpy as np
from dotenv import load_dotenv
import anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

client = anthropic.Anthropic()

MODEL = "claude-sonnet-4-20250514"

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"


## Implementation

We'll build the Memory-as-a-Tool system in four steps:

1. Define the four memory tool schemas.
2. Build a `MemoryStore` backend with embedding-based search.
3. Write a tool dispatcher that routes tool calls to memory operations.
4. Wire up the agentic loop that drives the whole system.


### Step 1: Define Memory Tool Schemas

Each tool schema tells the LLM what the tool does and what inputs it expects. Anthropic's format uses `input_schema` with standard JSON Schema. Clear descriptions guide the model on when to call each tool.


In [ ]:
memory_tools = [
    {
        "name": "save_memory",
        "description": (
            "Save an important fact, preference, or piece of information to "
            "long-term memory. Use this when the user shares something worth "
            "remembering for future conversations: personal details, preferences, "
            "goals, or corrections to previous facts."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "string",
                    "description": "The fact or information to remember.",
                },
                "tags": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Optional category tags (e.g. 'preference', 'personal', 'work').",
                },
            },
            "required": ["content"],
        },
    },
    {
        "name": "search_memory",
        "description": (
            "Search long-term memory for facts relevant to a query. Use this "
            "before answering questions that might benefit from previously stored "
            "context, such as user preferences or past conversations."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query describing what to look for.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Maximum number of results to return. Defaults to 5.",
                    "default": 5,
                },
            },
            "required": ["query"],
        },
    }
]

Now we define the `update_memory` and `delete_memory` tools.
These let the agent correct stale facts and remove information the user wants forgotten.
Together with `save_memory` and `search_memory`, they form a complete CRUD interface.

In [ ]:
memory_tools += [
    {
        "name": "update_memory",
        "description": (
            "Update an existing memory by its ID. Use this when the user corrects "
            "a previously stored fact or when information changes. Search for the "
            "memory first to get its ID."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "memory_id": {
                    "type": "string",
                    "description": "The ID of the memory to update.",
                },
                "new_content": {
                    "type": "string",
                    "description": "The updated fact or information.",
                },
            },
            "required": ["memory_id", "new_content"],
        },
    },
    {
        "name": "delete_memory",
        "description": (
            "Delete a memory by its ID. Use this when stored information is "
            "no longer true, relevant, or when the user asks you to forget something."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "memory_id": {
                    "type": "string",
                    "description": "The ID of the memory to delete.",
                },
            },
            "required": ["memory_id"],
        },
    },
]

print(f"Defined {len(memory_tools)} memory tools:")
for tool in memory_tools:
    print(f"  - {tool['name']}: {tool['description'][:60]}...")


### Step 2: Build the Memory Store

Think of a filing cabinet where each folder has a label written in invisible ink. To find the right folder, you describe what you're looking for, and a helper compares your description against every label. That's semantic search.

Our `MemoryStore` converts text into vectors (lists of numbers that capture meaning). When you search, we embed the query and find stored memories with the highest cosine similarity.

For this prototype, we use a lightweight hash-based embedding. In production, swap it for a real embedding model like `voyage-3`, `text-embedding-3-small`, or a sentence-transformer. Each memory gets a unique ID, a timestamp, optional tags, and its embedding vector.


In [ ]:
class MemoryStore:
    """Vector-backed memory store with CRUD operations."""

    def __init__(self):
        self.memories: dict[str, dict] = {}  # memory_id -> record

    def _simple_embed(self, text: str) -> list[float]:
        """Lightweight embedding using character-level hashing.

        This is a prototype. It captures word overlap reasonably well
        for short texts. For production, use a proper embedding model
        like voyage-3, text-embedding-3-small, or sentence-transformers.
        """
        dims = 128
        vec = np.zeros(dims)
        words = text.lower().split()
        for word in words:
            # Hash each word to a set of dimensions and increment
            h = hash(word)
            for i in range(3):  # each word activates 3 dimensions
                idx = abs(h + i * 7919) % dims
                vec[idx] += 1.0
        # Normalize to unit length
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec.tolist()

    def save(self, content: str, tags: list[str] | None = None) -> str:
        """Store a new memory. Returns the memory ID."""
        memory_id = str(uuid.uuid4())[:8]
        self.memories[memory_id] = {
            "id": memory_id,
            "content": content,
            "tags": tags or [],
            "created_at": datetime.now(timezone.utc).isoformat(),
            "embedding": self._simple_embed(content),
        }
        return memory_id

The `search` method is where semantic retrieval happens.
It embeds the query, then compares it against every stored memory using cosine similarity (a measure of how similar two vectors are).
Results come back ranked by relevance, with the best matches first.

In [ ]:
    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """Find memories most similar to the query."""
        if not self.memories:
            return []
        query_vec = np.array(self._simple_embed(query))
        scored = []
        for mem in self.memories.values():
            mem_vec = np.array(mem["embedding"])
            # Cosine similarity (vectors are already normalized)
            similarity = float(np.dot(query_vec, mem_vec))
            scored.append((similarity, mem))
        scored.sort(key=lambda x: x[0], reverse=True)
        results = []
        for score, mem in scored[:top_k]:
            results.append({
                "id": mem["id"],
                "content": mem["content"],
                "tags": mem["tags"],
                "similarity": round(score, 3),
                "created_at": mem["created_at"],
            })
        return results

The remaining CRUD methods handle updates, deletes, and listing.
When you update a memory, the embedding gets recalculated so future searches reflect the new content.
The `list_all` method strips out embeddings for clean display.

In [ ]:
    def update(self, memory_id: str, new_content: str) -> bool:
        """Update an existing memory's content. Returns True if found."""
        if memory_id not in self.memories:
            return False
        self.memories[memory_id]["content"] = new_content
        self.memories[memory_id]["embedding"] = self._simple_embed(new_content)
        return True

    def delete(self, memory_id: str) -> bool:
        """Remove a memory by ID. Returns True if found."""
        if memory_id not in self.memories:
            return False
        del self.memories[memory_id]
        return True

    def list_all(self) -> list[dict]:
        """Return all memories (without embeddings, for display)."""
        return [
            {"id": m["id"], "content": m["content"], "tags": m["tags"]}
            for m in self.memories.values()
        ]


# Quick test
store = MemoryStore()
test_id = store.save("Alice is a machine-learning engineer.", tags=["personal"])
print(f"Saved memory with ID: {test_id}")
results = store.search("What does Alice do for work?")
print(f"Search results: {json.dumps(results, indent=2)}")

### Step 3: Tool Dispatcher

The dispatcher is the bridge between the LLM's tool calls and the memory store. When the LLM emits a `tool_use` block, we read the tool name and inputs, call the right method on `MemoryStore`, and format the result as a string.


In [ ]:
def dispatch_tool(tool_name: str, tool_input: dict, store: MemoryStore) -> str:
    """Execute a memory tool call and return a string result."""
    if tool_name == "save_memory":
        memory_id = store.save(
            content=tool_input["content"],
            tags=tool_input.get("tags"),
        )
        return json.dumps({"status": "saved", "memory_id": memory_id})

    elif tool_name == "search_memory":
        results = store.search(
            query=tool_input["query"],
            top_k=tool_input.get("top_k", 5),
        )
        return json.dumps({"results": results, "count": len(results)})

    elif tool_name == "update_memory":
        success = store.update(
            memory_id=tool_input["memory_id"],
            new_content=tool_input["new_content"],
        )
        status = "updated" if success else "not_found"
        return json.dumps({"status": status})

    elif tool_name == "delete_memory":
        success = store.delete(memory_id=tool_input["memory_id"])
        status = "deleted" if success else "not_found"
        return json.dumps({"status": status})

    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})


# Quick test: dispatch a save call
result = dispatch_tool("save_memory", {"content": "Bob likes Thai food", "tags": ["preference"]}, store)
print(f"Dispatch result: {result}")


### Step 4: System Prompt

The system prompt guides the model's memory behavior. It tells the model when to save, search, update, and delete. Without clear instructions, the model might ignore the memory tools or use them at the wrong times.


In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant with long-term memory.

You have four memory tools available:

WHEN TO SAVE:
- Save when the user shares personal facts, preferences, or important details.
- Save corrections to previously known information.
- Do NOT save greetings, small talk, or temporary context.

WHEN TO SEARCH:
- Search before answering questions that might benefit from prior context.
- Search when the user asks "do you remember" or references past conversations.

WHEN TO UPDATE:
- Update when the user corrects a previously stored fact.
- Search first to find the memory ID, then update it.

WHEN TO DELETE:
- Delete when stored information is explicitly wrong or the user asks you to forget.

Keep responses concise. When you use memory tools, briefly mention what you did
(e.g. "I've saved that to memory" or "Let me check my memory")."""

print("System prompt configured.")


### Step 5: The Agentic Tool-Use Loop

Think of a phone call where you can put someone on hold to look something up. The caller (user) asks a question. You (the agent) pause the conversation, check your files, come back, and answer. If you need more information, you can pause again. The call ends when you give a final answer.

The agentic loop works the same way. The LLM generates a response. If it includes a `tool_use` block, we execute the tool and feed the result back. The loop continues until the model produces a final `end_turn` response.


In [ ]:
def chat_with_memory(
    user_message: str,
    messages: list[dict],
    store: MemoryStore,
) -> tuple[str, list[dict]]:
    """Send a message through the agentic loop with memory tools.

    Returns the assistant's final text and the updated message list.
    """
    messages.append({"role": "user", "content": user_message})

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=memory_tools,
            messages=messages,
        )

        # Add the assistant's full response to history
        messages.append({"role": "assistant", "content": response.content})

        # If the model didn't call any tools, we're done
        if response.stop_reason == "end_turn":
            # Extract the final text from the response
            final_text = ""
            for block in response.content:
                if block.type == "text":
                    final_text += block.text
            return final_text, messages

        # Process each tool call in the response
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  [tool] {block.name}({json.dumps(block.input)[:80]})")
                result = dispatch_tool(block.name, block.input, store)
                print(f"  [result] {result[:100]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        # Feed tool results back to the model
        messages.append({"role": "user", "content": tool_results})


print("Agentic loop ready.")


## Example Run

Let's run a multi-turn conversation that exercises all four memory operations. We'll:
1. Share some personal facts (the agent should save them).
2. Ask a question that requires memory recall (the agent should search).
3. Correct a fact (the agent should update).
4. Ask the agent to forget something (the agent should delete).


In [ ]:
# Fresh memory store and conversation
store = MemoryStore()
messages = []

exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer at Anthropic.",
    "I'm allergic to peanuts and I prefer vegetarian food.",
    "What do you remember about me?",
    "Actually, I moved to a new company. I now work at DeepMind.",
    "Please forget about my food allergy. I'd rather not have that stored.",
    "What do you know about me now?",
]

for user_msg in exchanges:
    print(f"\n{'='*60}")
    print(f"User: {user_msg}")
    print("-" * 60)
    reply, messages = chat_with_memory(user_msg, messages, store)
    print(f"\nAssistant: {reply}")


Let's inspect what's in the memory store after this conversation.


In [ ]:
print("Current memory store contents:")
print("-" * 40)
for mem in store.list_all():
    print(f"  [{mem['id']}] {mem['content']}")
    if mem['tags']:
        print(f"           tags: {mem['tags']}")
print(f"\nTotal memories: {len(store.memories)}")


### Tool Chaining

One strength of this pattern is that the agent can chain tool calls in a single turn. For example, it might search for an old fact, find it's outdated, and update it. All before sending a final response. Let's see this in action.


In [ ]:
# Continue the same conversation
user_msg = "Actually, my name changed too. I go by Ali now, not Alice."
print(f"User: {user_msg}")
print("-" * 60)
reply, messages = chat_with_memory(user_msg, messages, store)
print(f"\nAssistant: {reply}")

print("\nMemory store after update:")
for mem in store.list_all():
    print(f"  [{mem['id']}] {mem['content']}")


## Tradeoffs

### When Memory-as-a-Tool Works Well

- **Selective memory**: The agent stores only what matters. A food preference gets saved. "How's the weather?" does not. This keeps the memory store clean and relevant.
- **Self-correcting**: The agent can search, find stale facts, and update them in one turn. Pipeline-based memory systems often store duplicates or contradictions.
- **Transparent**: Every memory operation shows up as a tool call. You can log, audit, and debug exactly what the agent remembered and when.
- **Composable**: Memory tools sit alongside other tools. The agent can search memory, call a weather API, and save the result. All in one turn.

### When It Breaks Down

- **LLM reliability**: The agent might forget to save something important. Or it might save too much. The quality of memory management depends on the model's judgment.
- **Latency**: Each tool call adds a round trip. If the agent chains three memory operations before answering, that's three extra API calls. This adds latency (the delay between the user's question and the final answer).
- **Token cost**: Tool definitions consume tokens in every request. Four tool schemas add roughly 500-800 tokens to each call. For short conversations, this overhead may not be worth it.
- **No background processing**: The agent only manages memory during active turns. It won't consolidate or clean up memories between conversations unless you build that separately.


## Further Reading

- [Anthropic Tool Use Documentation](https://docs.anthropic.com/en/docs/build-with-claude/tool-use?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Official guide to defining tools, processing `tool_use`/`tool_result` blocks, and building agentic loops with Claude.
- [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): OpenAI's equivalent pattern for exposing callable functions to GPT models during generation.
- [Packer et al., "MemGPT: Towards LLMs as Operating Systems," 2023 (arXiv:2310.08560)](https://arxiv.org/abs/2310.08560): The foundational paper on treating memory management as agent actions. Describes explicit read/write tools for a virtual memory hierarchy.
- [Anthropic: Building Effective Agents (2025)](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Architectural patterns for production agents, including tool design principles.
- [Zep Memory Tools Integration](https://docs.getzep.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): A production memory service that exposes memory as tool-callable APIs with automatic fact extraction.


*\u2190 Previous: [22 - Multi-Agent Shared Memory](../22_multi_agent_shared_memory/) \u00b7 Next: [24 - Graph Memory with Graphiti](../24_graph_memory_graphiti/) \u2192*


## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Memory tagging tool
Add a fifth tool, `tag_memory`, to `memory_tools` that lets the agent assign a topic label to a stored memory. Update `dispatch_tool()` and `MemoryStore` to support tags. Run a conversation and verify the agent uses tags to categorize its memories.

### Challenge 2: Tool usage frequency analysis
Run a 20-turn conversation with `chat_with_memory()`. Count how many times the agent calls each tool (save, search, update, delete). Print a summary table. Identify which turns trigger memory saves vs. searches and whether the agent ever updates or deletes memories unprompted.

### Challenge 3: Agent-controlled vs. automatic memory
Run the same 15-turn conversation twice: once with the tool-based approach from this notebook, and once with automatic entity extraction from 07 Entity Memory. After each run, ask 10 recall questions. Compare which approach stores more relevant facts and which answers more questions correctly.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--23-memory-with-tools--memory-with-tools)
